<!-- SPDX-License-Identifier: Apache-2.0 -->
# FoaMES agent benchmark on a free GPU (Kaggle or Colab)

Runs the open-weight models that are too big for a GitHub-hosted runner (Qwen3-Coder 30B, Devstral Small 24B, gpt-oss 20b, and any other Ollama model you list) through the benchmark's pure tasks, with aider as the harness, the same as the Agent benchmark workflow.

**Kaggle (recommended):** New Notebook, File, Import Notebook, paste this file's GitHub URL. Settings: Accelerator **GPU T4 x2**, Internet **on** (needs a phone-verified account). Then **Save Version, Save & Run All (Commit)**: it runs headless for up to 12 hours. When it finishes, open the version, Output, and download `foames-bench-results.zip`.

**Colab:** Runtime, Change runtime type, T4 GPU, then Run all. Keep the tab open; free sessions stop when idle. Download the zip from the Files pane.

Send the zip back (drop it in the project thread). `publish_results.py` merges it with the runner results; the pages say where each model ran.

**Sandbox.** Notebooks have no Docker. The harness uses its `unshare` sandbox when the notebook allows user namespaces, and otherwise runs with `--sandbox none`, which the report marks: in that case the agent could in principle read the hidden tests, so treat those numbers as indicative until rerun sandboxed. Nothing private goes anywhere: the tasks are public and use invented data, and the models run on the notebook's own GPU.

In [ ]:
# Settings: models (Ollama tags), tasks (empty for all pure tasks), repetitions, FoaMES ref.
MODELS = ["qwen3-coder:30b", "devstral:24b", "gpt-oss:20b"]
TASKS = []          # e.g. ["counter-reset", "oee-unknown-time"]
REPS = 1
FOAMES_REF = "main"
AIDER_VERSION = "0.86.1"

In [ ]:
import os, subprocess, json, shutil, datetime, pathlib
def sh(cmd, check=True):
    print("$", cmd); r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-4000:], r.stderr[-2000:]); 
    if check and r.returncode: raise SystemExit(r.returncode)
    return r.stdout
WORK = pathlib.Path("/kaggle/working") if pathlib.Path("/kaggle/working").exists() else pathlib.Path("/content")
ENV = "Kaggle notebook" if "kaggle" in str(WORK) else "Colab notebook"
sh("nvidia-smi --query-gpu=name,memory.total --format=csv", check=False)
sh(f"rm -rf {WORK}/foames && git clone --depth 1 --branch {FOAMES_REF} https://github.com/CLR-Technologies-Inc/foames.git {WORK}/foames")
sh(f"pip install -q aider-chat=={AIDER_VERSION}")
sh("curl -fsSL https://ollama.com/install.sh | sh")
subprocess.Popen("ollama serve > /tmp/ollama.log 2>&1", shell=True)
import time; time.sleep(8)
sh("ollama --version")

In [ ]:
BENCH = WORK / "foames" / "benchmarks"
gpu = sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", check=False).strip().replace("\n", " + ")
# Which sandbox works here: unshare if user namespaces are allowed, otherwise none (marked in the report).
probe = subprocess.run("unshare --user --map-root-user --mount --pid --fork true", shell=True)
SANDBOX = "unshare" if probe.returncode == 0 else "none"
print("sandbox:", SANDBOX)
cfg = json.loads((BENCH / "agents.json").read_text())
names = []
for tag in MODELS:
    name = "aider-ollama-" + tag.replace(":", "-")
    if not any(a["name"] == name for a in cfg["agents"]):
        base = next(a for a in cfg["agents"] if a["name"] == "aider-ollama-qwen2.5-coder-7b")
        cfg["agents"].append({**base, "name": name, "model": "ollama_chat/" + tag,
                              "notes": f"Aider with {tag} served by Ollama on the notebook GPU."})
    names.append(name)
agents_file = WORK / "agents.notebook.json"
agents_file.write_text(json.dumps(cfg))

In [ ]:
OUT = WORK / "runs"
OUT.mkdir(exist_ok=True)
os.environ["OLLAMA_API_BASE"] = "http://127.0.0.1:11434"
for tag, name in zip(MODELS, names):
    sh(f"ollama pull {tag}")
    digest = [l.split()[1] for l in sh("ollama list").splitlines()[1:] if l.split() and l.split()[0] == tag]
    rd = OUT / name
    cmd = (f"cd {BENCH} && python -m foames_bench --agents-file {agents_file} run --agents {name} --reps {REPS} "
           f"--sandbox {SANDBOX} --out {rd} " + (f"--tasks {' '.join(TASKS)}" if TASKS else ""))
    sh(cmd, check=False)
    rd.mkdir(exist_ok=True)
    (rd / "environment.txt").write_text(f"{ENV} ({gpu}), sandbox {SANDBOX}")
    (rd / "model-digest.txt").write_text(f"Ollama {tag} digest {digest[0] if digest else '?'}, pulled {datetime.date.today()}")
    (rd / "aider-version.txt").write_text(sh("aider --version").strip())
    (rd / "ollama-version.txt").write_text(sh("ollama --version").strip())
    sh(f"ollama rm {tag}", check=False)   # free the disk for the next model

In [ ]:
sh(f"cd {BENCH} && python scripts/publish_results.py --runs {OUT} --out-root {WORK}/report", check=False)
shutil.make_archive(str(WORK / "foames-bench-results"), "zip", WORK, "runs")
print("Download", WORK / "foames-bench-results.zip")